<a href="https://www.kaggle.com/code/suneku/voice-changer-public?scriptVersionId=203548435" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

### [w-okada's Voice Changer](https://github.com/deiteris/voice-changer) | **Kaggle**

---

## **⬇ VERY IMPORTANT ⬇**

You can use the following settings for optimal results:

Best performance: `f0: fcpe | Chunk: 56.0ms or higher | Extra: 2.7s`<br>
Best quality: `f0: rmvpe | Chunk: 64.0ms or higher | Extra: 5s`<br>
**Don't forget to select a GPU in the GPU field, <b>NEVER</b> use CPU!

You can tune `Chunk` for lower/higher delay and `Extra` for better quality.

---
**Credits**<br>
Realtime Voice Changer by [w-okada](https://github.com/w-okada)<br>
Original instructions by [Hina](https://github.com/HinaBl)<br>

**Need help?** [AI Hub Discord](https://discord.gg/aihub) » ***#help-realtime-vc***

---

# Kaggle Tutorial
Running this notebook can be a bit complicated.\
After created your Kaggle account, you'll need to **verify your phone number** to be able to use Internet Connection and GPUs.

Follow the instructions on the image below and don't forget to set **PERSISTENCE** to **Files only**.

![instructions.png](https://i.imgur.com/0NutkD8.png)

# Clone repository and install dependencies
This first step will download the latest version of Voice Changer and install the dependencies. **It will take some time to complete.**

In [ ]:
# This will make that we're on the right folder before installing
%cd /kaggle/working/

# To workaround an issue with METADATA not found for aiohttp
# !conda install -y aiohttp
!pip install colorama python-dotenv pyngrok --quiet

from colorama import Fore, Style
import requests, os

print(f"{Fore.CYAN}> Downloading prebuilt executable...{Style.RESET_ALL}")

import re
import subprocess

res = requests.get('https://api.github.com/repos/deiteris/voice-changer/releases/latest', timeout=30)
res.raise_for_status()
release_info = res.json()
ARCHIVE = "voice-changer-linux-amd64-cuda.tar.gz"

from pathlib import Path
from tempfile import TemporaryDirectory, mkdtemp
import shutil

def download_and_unpack(assets, staging, archive):
    matches = {asset["name"]: asset for asset in assets
               if asset["name"] == archive or re.fullmatch(re.escape(archive) + r"\.[a-z]{2}", asset["name"])}
    names = [archive] if archive in matches else sorted(matches)
    if not names or (names != [archive] and names != [
        archive + "." + chr(97 + i // 26) + chr(97 + i % 26) for i in range(len(names))
    ]):
        raise RuntimeError("No complete CUDA archive found in the selected release.")
    with subprocess.Popen(["tar", "-xzf", "-", "-C", str(staging)], stdin=subprocess.PIPE) as extract:
        with extract.stdin:
            for name in names:
                with requests.get(matches[name]["browser_download_url"], stream=True, timeout=60) as download:
                    download.raise_for_status()
                    for chunk in download.iter_content(chunk_size=1024 * 1024):
                        extract.stdin.write(chunk)
        if extract.wait() != 0:
            raise RuntimeError("Failed to extract the release archive.")


def install_runtime(assets, install, executable, archive):
    with TemporaryDirectory(prefix=".vc-download-", dir=install.parent) as staging:
        source = Path(staging) / "MMVCServerSIO"
        download_and_unpack(assets, source.parent, archive)
        binary = source / "MMVCServerSIO"
        internal = source / "_internal"
        if not binary.is_file() or not os.access(binary, os.X_OK) or not (internal / "version.txt").is_file():
            raise RuntimeError("The downloaded release is missing its executable or version file.")

        install.mkdir(exist_ok=True)
        backup = Path(mkdtemp(prefix=".vc-backup-", dir=install.parent))
        saved, replaced = [], []
        try:
            for incoming, name in ((binary, executable), (internal, "_internal")):
                destination = install / name
                if destination.exists() or destination.is_symlink():
                    destination.rename(backup / name)
                    saved.append(name)
                incoming.rename(destination)
                replaced.append((destination, incoming))
        except (OSError, KeyboardInterrupt):
            for destination, incoming in reversed(replaced):
                destination.rename(incoming)
            for name in reversed(saved):
                (backup / name).rename(install / name)
            backup.rmdir()
            raise
        else:
            shutil.rmtree(backup)


install = Path('/kaggle/working/MMVCServerSIO')
try:
    local_version = (install / '_internal' / 'version.txt').read_text().strip()
except (OSError, UnicodeError):
    local_version = None

remote_version = release_info['tag_name']
executable = install / 'VoiceChanger'
if local_version != remote_version or not executable.is_file() or not os.access(executable, os.X_OK):
    print(f'Installing {remote_version}...')
    install_runtime(release_info['assets'], install, 'VoiceChanger', ARCHIVE)
else:
    print(f'{remote_version} is already installed.')

%cd MMVCServerSIO

print(f"{Fore.CYAN}> Installing libportaudio2...{Style.RESET_ALL}")
!apt-get -y install libportaudio2 -qq
print(f"{Fore.CYAN}> Done! Proceed with the next steps.{Style.RESET_ALL}")

# Migrate the data

This cell will migrate your models from old version to new version (if present).

In [ ]:
from pathlib import Path

legacy = Path('/kaggle/working/Hmod/server')
install = Path('/kaggle/working/MMVCServerSIO')
moves = [(legacy / name, install / name) for name in ('model_dir', 'pretrain')
         if (legacy / name).exists()]
for source, destination in moves:
    if not source.is_dir() or source.is_symlink():
        raise ValueError(f'Expected a directory at {source}')
    if destination.exists() or destination.is_symlink():
        raise FileExistsError(f'Migration would overwrite {destination}. Legacy files are preserved.')
for source, destination in moves:
    source.rename(destination)
    print(f'Migrated {destination.name} to {destination}')

# Configure the server

This cell will apply necessary configuration to the server.

In [ ]:
%cd /kaggle/working/MMVCServerSIO

from dotenv import set_key

set_key('.env', "SAMPLE_MODE", "")

Ready = True

# Start Server **using ngrok**
This cell will start the server, the first time that you run it will download the models, so it can take a while (~1-2 minutes)

---
You'll need a ngrok account, but <font color=green>**it's free**</font> and easy to create!
---
**1** - Create a **free** account at [ngrok](https://dashboard.ngrok.com/signup)\
**2** - If you didn't logged in with Google or Github, you will need to **verify your e-mail**!\
**3** - Click [this link](https://dashboard.ngrok.com/get-started/your-authtoken) to get your auth token, and replace **TOKEN_HERE** with your token.\
**4** - *(optional)* Change to a region near to you

In [ ]:
Token = 'TOKEN_HERE'
Region = "eu" # Read the instructions below
ClearConsole = True

# You can change the region for a better latency, use only the abbreviation
# Choose between this options:
# us -> United States (Ohio)
# ap -> Asia/Pacific (Singapore)
# au -> Australia (Sydney)
# eu -> Europe (Frankfurt)
# in -> India (Mumbai)
# jp -> Japan (Tokyo)
# sa -> South America (Sao Paulo)

# ---------------------------------
# DO NOT TOUCH ANYTHING DOWN BELOW!

%cd /kaggle/working/MMVCServerSIO

if not globals().get('Ready', False):
    print("Go back and run first and second cells.")
else:
    from pyngrok import conf, ngrok
    MyConfig = conf.PyngrokConfig()
    MyConfig.auth_token = Token
    MyConfig.region = Region[0:2]
    conf.get_default().authtoken = Token
    conf.get_default().region = Region
    conf.set_default(MyConfig);

    import threading, time, socket
    PORT = 18888

    import json
    from pyngrok import ngrok
    from IPython.display import clear_output

    ngrokConnection = ngrok.connect(PORT)
    public_url = ngrokConnection.public_url
    set_key('.env', "ALLOWED_ORIGINS", json.dumps([public_url]))

    def wait_for_server():
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', PORT))
            if result == 0:
                break
            sock.close()
        if ClearConsole:
            clear_output()
        print("--------- SERVER READY! ---------")
        print("Your server is available at:")
        print(public_url)
        print("---------------------------------")

    threading.Thread(target=wait_for_server, daemon=True).start()
    
    !chmod +x ./VoiceChanger
    
    !./VoiceChanger

    ngrok.disconnect(ngrokConnection.public_url)